In [1]:
import sys, os
import json
sys.path.append(os.getcwd() + "/FredScript/src")

from openrocket_parser.core import load_rocket_from_xml

filePath = os.getcwd() + "/ReferencedFiles/rocketpyillionaire_grindset.ork"

(rocket, rocketpyMapping) = load_rocket_from_xml(filePath)

print(rocketpyMapping)

rocketLength = 0
rocketLength += rocketpyMapping["nose_cone_length"]
rocketLength += rocketpyMapping["boattail_length"]
rocketMass = 0

firstRailId = -1
secondRailId = -1
firstRailSeen = False
secondRailSeen = False

parachuteIds = []

def getIdFromKey(key):
    id = key[key.index("_") + 1:]
    id = id[:id.index("_")]
    print(id)
    return id

for key in rocketpyMapping:
    if key.startswith("bodyTube") and key.endswith("length"):
        rocketLength += rocketpyMapping[key]
    if "mass" in key:
        rocketMass += rocketpyMapping[key]
    
    if "rail" in key:
        print(key)
        railId = getIdFromKey(key)
        if not firstRailSeen:
            firstRailSeen = True
            firstRailId = railId
        elif railId != firstRailId and not secondRailSeen:
            secondRailId = railId
            secondRailSeen = True

    if "parachute" in key:
        print(key)
        parachuteId = getIdFromKey(key)
        if parachuteId not in parachuteIds:
            parachuteIds.append(parachuteId)

    

rocketpyMapping["spLength"] = rocketLength
rocketpyMapping["spMass"] = rocketMass
print(f"Final length: {rocketLength}")
print(f"Final mass: {rocketMass}")

rocketpyMapping["bottailPos"] = rocketpyMapping["spLength"] - rocketpyMapping["boattail_length"]

if rocketpyMapping["fin_position_version"] == "bottom":
    rocketpyMapping["fin_position"] = rocketpyMapping["spLength"] + rocketpyMapping["fin_position"]

print(parachuteIds)


if rocketpyMapping[f"rail_{firstRailId}_positionType"] == "middle":
    firstRailPos = rocketpyMapping[f"rail_{firstRailId}_position"] + rocketpyMapping["spLength"] / 2

if rocketpyMapping[f"rail_{secondRailId}_positionType"] == "middle":
    secondRailPos = rocketpyMapping[f"rail_{secondRailId}_position"] + rocketpyMapping["spLength"] / 2

if firstRailPos < secondRailPos:
    rocketpyMapping["lower_railbutton_position"] = firstRailPos
    rocketpyMapping["upper_railbutton_position"] = secondRailPos
else:
    rocketpyMapping["lower_railbutton_position"] = secondRailPos
    rocketpyMapping["upper_railbutton_position"] = firstRailPos

rocketpyMapping["railbutton_angular_position"] = (rocketpyMapping[f"rail_{firstRailId}_angle"] + rocketpyMapping[f"rail_{secondRailId}_angle"]) / 2

rocketpyMapping["parachutes"] = {}

for id in parachuteIds:
    rocketpyMapping["parachutes"][id] = {
        "lag": rocketpyMapping[f"parachute_{id}_lag"],
        "cd": rocketpyMapping[f"parachute_{id}_cd"],
        "trigger": rocketpyMapping[f"parachute_{id}_trigger"],
        "name": id
    }

with open ("ReferencedFiles/RelevantOpenRocket.json", "w") as f:
    json.dump(rocketpyMapping, f, indent=4)

Registering subcomponent → <class 'openrocket_parser.components.components.Subcomponent'>
Registering bulkhead → <class 'openrocket_parser.components.components.Bulkhead'>
Registering shockcord → <class 'openrocket_parser.components.components.ShockCord'>
Registering tubecoupler → <class 'openrocket_parser.components.components.TubeCoupler'>
Registering parachute → <class 'openrocket_parser.components.components.Parachute'>
Registering railbutton → <class 'openrocket_parser.components.components.RailButton'>
Registering motorconfiguration → <class 'openrocket_parser.components.components.MotorConfig'>
Registering masscomponent → <class 'openrocket_parser.components.components.MassComponent'>
Registering innertube → <class 'openrocket_parser.components.components.InnerTube'>
Registering trapezoidfinset → <class 'openrocket_parser.components.components.TrapezoidFinSet'>
Registering centeringring → <class 'openrocket_parser.components.components.CenteringRing'>
Registering transition → <c